# Chemistry MCQ pilot — two-page Surya OCR

Run in a fresh Colab **T4 GPU** runtime, top to bottom. First push the feature branch containing this notebook/config. This uses a separate Drive store and preserves source hashes, page geometry and unverified status. It does not create or approve MCQ training rows. Model setup may take several minutes.


In [ ]:
from pathlib import Path
import json, subprocess, sys, shutil, datetime
from google.colab import drive

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU, then reconnect.')
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive')
ROOT = Path('/content/sinhala-corpus')
STORE = Path('/content/drive/MyDrive/sinhala-corpus/mcq_pilots/chemistry_2021_v1')
BRANCH = 'feat/mcq-answer-matching'
if ROOT.exists():
    raise RuntimeError('Repository already exists. Use a fresh runtime for setup; resume processing using the later cells.')
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',
                'https://github.com/Warusha-Shamalka/sinhala-corpus.git',str(ROOT)],check=True)


## Install the existing pinned Surya environment


In [ ]:
ENV = Path('/content/corpus-env')
if ENV.exists():
    raise RuntimeError('Use a fresh runtime instead of replacing an existing environment.')
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','venv','--python','3.12.12','--seed',str(ENV)],check=True)
PYTHON = str(ENV/'bin/python')
def pip(*args):
    subprocess.run([PYTHON,'-m','pip','install',*args],check=True)
pip('torch==2.7.1','torchvision==0.22.1','--index-url','https://download.pytorch.org/whl/cu126')
pip('PyYAML==6.0.3','surya-ocr==0.17.1','transformers==4.57.6','Pillow==10.4.0')
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','poppler-utils'],check=True)


In [ ]:
import csv, hashlib
PILOT = json.loads((ROOT/'configs/pilots/chemistry_2021.json').read_text())
def execute(*args):
    result = subprocess.run([PYTHON,'-B','-m','pipeline.workflow',*args],cwd=ROOT,text=True,stdout=subprocess.PIPE)
    print(result.stdout)
    result.check_returncode()
    return json.loads(result.stdout)
CONFIG = STORE/'configs/pipeline.yaml'
if not STORE.exists():
    fields = next(csv.reader((ROOT/'corpus/catalog/documents.csv').open()))
    seed = Path('/content/chemistry_pilot_seed.csv')
    with seed.open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader()
        for job in PILOT['jobs']:
            row={field:'' for field in fields}
            row.update(doc_id=job['doc_id'],title=job['title'],source='AlApiEdu',source_url=job['source_url'],
                       document_type='past_paper' if job['role']=='paper' else 'marking_scheme',status='DISCOVERED')
            writer.writerow(row)
    execute('init','--store',str(STORE),'--seed',str(seed))
if not CONFIG.is_file():
    raise RuntimeError('Pilot store has no configuration; inspect it before continuing.')
print('Persistent pilot store:', STORE)


## Download only the two selected public PDFs and check pinned hashes


In [ ]:
configure_download = r"""
import json,sys,yaml
from pathlib import Path
p=Path(sys.argv[1]);cfg=yaml.safe_load(p.read_text())
cfg['download'].update(max_documents=2,max_retries=0,max_requests=8,max_seconds=120)
p.write_text(yaml.safe_dump(cfg,sort_keys=False));print(json.dumps(cfg))
"""
cfg=json.loads(subprocess.check_output([PYTHON,'-c',configure_download,str(CONFIG)],text=True))
ids=[part for job in PILOT['jobs'] for part in ['--doc-id',job['doc_id']]]
execute('collect','--config',str(CONFIG),*ids,'--progress','plain')
catalog={row['doc_id']:row for row in csv.DictReader(Path(cfg['download']['catalog_path']).open())}
for job in PILOT['jobs']:
    row=catalog[job['doc_id']]
    if row['sha256'] != job['raw_sha256'] or row['source_url'] != job['source_url']:
        raise RuntimeError('Source identity changed: '+job['doc_id'])
    raw=Path(row['local_filename'])
    if not raw.is_absolute(): raw=ROOT/raw
    if hashlib.sha256(raw.read_bytes()).hexdigest() != job['raw_sha256']:
        raise RuntimeError('Raw bytes changed: '+job['doc_id'])
print('Both source hashes match the local pilot.')


## Prepare and freeze Surya weights


In [ ]:
MODEL_PATH = Path('/content/surya-models')
selection_code = r"""
import json,sys
from pathlib import Path
from pipeline.ocr.surya_backend import prepare_models,verify_models
from pipeline.download.downloader import file_hash
root=Path(sys.argv[1]); output=Path(sys.argv[2])
if root.exists():
    revision=file_hash(root/'selection.json')
    result={**verify_models(root,revision),'path':str(root),'revision':revision,
            'revision_kind':'local_components_manifest_sha256'}
else:
    result=prepare_models(root)
output.write_text(json.dumps(result,indent=2)+'\n')
"""
SELECTION = Path('/content/corpus_model_selection.json')
subprocess.run([PYTHON,'-c',selection_code,str(MODEL_PATH),str(SELECTION)],cwd=ROOT,check=True)
selection = json.loads(SELECTION.read_text())
manifest_root = STORE/'model_manifests'
manifest_root.mkdir(exist_ok=True)
model_manifest = manifest_root/(selection['revision']+'.json')
if model_manifest.exists():
    if json.loads(model_manifest.read_text()) != selection:
        raise RuntimeError('Saved model manifest differs; inspect it before continuing.')
else:
    model_manifest.write_text(json.dumps(selection,indent=2)+'\n')
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
(manifest_root/f'requirements_{stamp}.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
(manifest_root/f'repository_commit_{stamp}.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True))


In [ ]:
configure_processing = r"""
import sys,yaml
from pathlib import Path
p=Path(sys.argv[1]);cfg=yaml.safe_load(p.read_text())
cfg['processing']['ocr'].update(backend='surya',model_path=sys.argv[2],revision=sys.argv[3])
cfg['processing'].update(max_pages=1,max_documents=1,ocr_mode='all')
p.write_text(yaml.safe_dump(cfg,sort_keys=False))
"""
subprocess.run([PYTHON,'-c',configure_processing,str(CONFIG),str(MODEL_PATH),selection['revision']],check=True)
reports=[]
for job in PILOT['jobs']:
    print('Processing',job['role'],job['doc_id'],'PDF page',job['page_number'],flush=True)
    report=execute('process','--config',str(CONFIG),'--doc-id',job['doc_id'],
                   '--page-number',str(job['page_number']),'--progress','plain')
    reports.append(report)


## Save an OCR handoff for the next matching step

This checks bundle integrity and records incomplete predictions. It preserves raw Surya word/line geometry for the key table. No manual Sinhala typing is needed.


In [ ]:
handoff_code = r"""
import json,sys
from pathlib import Path
from pipeline.processing.pages import verify_page
from pipeline.download.downloader import file_hash
reports=json.loads(Path(sys.argv[1]).read_text()); jobs=json.loads(Path(sys.argv[2]).read_text())['jobs']
items=[]
for job,report in zip(jobs,reports):
    matches=[r for r in report['results'] if r.get('doc_id')==job['doc_id'] and r.get('page_number')==job['page_number'] and r.get('bundle')]
    if len(matches)!=1: raise ValueError('Expected one page bundle for '+job['doc_id'])
    folder=Path(matches[0]['bundle']); m=json.loads((folder/'manifest.json').read_text())
    page=verify_page(folder,m['identity'])
    if page['identity']['raw_sha256']!=job['raw_sha256']: raise ValueError('Unexpected raw hash')
    items.append({'doc_id':job['doc_id'],'role':job['role'],'page_number':job['page_number'],
        'bundle':str(folder),'manifest_sha256':file_hash(folder/'manifest.json'),
        'outcome':page['prediction']['outcome'],'completion_status':page['prediction']['completion_status'],
        'quality_flags':page['quality_flags'],'training_approved':False})
result={'schema_version':'mcq-ocr-handoff-1','pages':items,'pairing_verified':False,'training_approved':False,
        'next_step':'Inspect OCR layout, extract question blocks and key table cells; keep ambiguity unresolved.'}
Path(sys.argv[3]).write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
"""
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN=STORE/'pilot_runs'/stamp
RUN.mkdir(parents=True)
(RUN/'reports.json').write_text(json.dumps(reports,indent=2)+'\n')
(RUN/'pilot.json').write_text(json.dumps(PILOT,indent=2)+'\n')
subprocess.run([PYTHON,'-c',handoff_code,str(RUN/'reports.json'),str(RUN/'pilot.json'),str(RUN/'handoff.json')],cwd=ROOT,check=True)
print((RUN/'handoff.json').read_text())
print('OCR handoff:',RUN/'handoff.json')
print('Next: share this handoff and the two page.json files so matching can be completed.')


## Download the handoff
Upload this ZIP back to the project conversation. It contains the two OCR page bundles and their hashes, so no manual Sinhala transcription is needed.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
handoff=json.loads((RUN/'handoff.json').read_text())
ARCHIVE=RUN/'pilot_ocr_handoff.zip'
with ZipFile(ARCHIVE,'x',compression=ZIP_DEFLATED) as archive:
    archive.write(RUN/'handoff.json','handoff.json')
    archive.write(RUN/'pilot.json','pilot.json')
    for page in handoff['pages']:
        folder=Path(page['bundle'])
        for name in ('manifest.json','page.json','text.txt','page.png'):
            archive.write(folder/name,f"{page['doc_id']}/page_{page['page_number']:06d}/{name}")
print('Saved:',ARCHIVE)
files.download(str(ARCHIVE))
